In [24]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
#import rasterio
# osx | tk
%matplotlib tk 

In [2]:
%matplotlib --list

Available matplotlib backends: ['agg', 'auto', 'cairo', 'gtk3', 'gtk3agg', 'gtk3cairo', 'gtk4', 'gtk4agg', 'gtk4cairo', 'inline', 'macosx', 'nbagg', 'notebook', 'osx', 'pdf', 'pgf', 'ps', 'qt', 'qt5', 'qt5agg', 'qt5cairo', 'qt6', 'qtagg', 'qtcairo', 'svg', 'template', 'tk', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wx', 'wxagg', 'wxcairo']


In [2]:
ls ~/Documents/GDrive/JBB/Reporte_2025/Artropofauna/'Anexo 8 Datos Darwin Core Consolidados.xlsx'

'/home/nelson/Documents/GDrive/JBB/Reporte_2025/Artropofauna/Anexo 8 Datos Darwin Core Consolidados.xlsx'


In [3]:
fname = '/home/nelson/Documents/GDrive/JBB/Reporte_2025/Artropofauna/Anexo 8 Datos Darwin Core Consolidados.xlsx'

In [4]:
artr = pd.read_excel(fname)

In [6]:
print(artr.iloc[0].to_string())

gbifID                                                                        3902035418
accessRights                                                                         NaN
bibliographicCitation                                                                NaN
language                                                                             NaN
license                                                                     CC_BY_NC_4_0
modified                                                            2022-08-19T00:49:47Z
publisher                                                                iNaturalist.org
references                             https://www.inaturalist.org/observations/69470815
rightsHolder                                                             Loreta Rosselli
type                                                                                 NaN
institutionID                                                                        NaN
collectionID         

In [5]:
artr.groupby('taxonomicStatus', dropna=False).size()

taxonomicStatus
ACCEPTED    27198
DOUBTFUL      134
SYNONYM       642
NaN          1582
dtype: int64

In [6]:
artr.groupby('taxonRank', dropna=False).size()

taxonRank
CLASS           262
FAMILY        10881
FORM              6
GENUS          3160
ORDER          2218
PHYLUM           38
SPECIES       12580
SUBSPECIES      311
UNRANKED        100
dtype: int64

In [7]:
artr.loc[
	(artr.decimalLatitude == '4,761111 -74,103611'),
	['decimalLatitude', 'decimalLongitude']] = [4.761111, -74.103611]

In [8]:
artr['decimalLatitude'] = artr.decimalLatitude.astype(float)
artr['decimalLongitude'] = artr.decimalLongitude.astype(float)

In [9]:
artr.loc[artr.decimalLongitude > -70, ['decimalLatitude', 'decimalLongitude']]

,decimalLatitude,decimalLongitude
29474,4.0,6032.0


In [10]:
artr[['decimalLatitude', 'decimalLongitude']].describe()

,decimalLatitude,decimalLongitude
count,29555.000000,29555.000000
mean,4.654846,-73.885819
std,0.101330,35.517980
min,3.742986,-74.663400
25%,4.610147,-74.111532
50%,4.664600,-74.083300
75%,4.704767,-74.063861
max,4.855570,6032.000000


In [11]:
dat = artr.loc[
	(artr.taxonomicStatus == 'ACCEPTED') &
	(artr.taxonRank == 'SPECIES') & 
	(artr.decimalLongitude < -70 ),
	['scientificName', 'decimalLongitude', 'decimalLatitude']
]

In [12]:
gdat = gpd.GeoDataFrame(
	dat, 
	geometry=gpd.points_from_xy(dat.decimalLongitude, dat.decimalLatitude),
	crs=4326
)

In [13]:
gdat.plot()

<Axes: >

In [30]:
locs = gpd.read_file("shared/loca/Loca.shp")
locs = locs.to_crs(4326)

In [31]:
greg = gdat.sjoin(locs, how='left', predicate='within')

In [32]:
locs = locs.merge(
	greg.groupby(['LocNombre', 'scientificName']
		).size(
		).reset_index(
		).groupby('LocNombre'
		).size(
		).reset_index(
		).rename(columns={0: 'Riqueza'}),
	how='left',
	on='LocNombre'
)

In [33]:
locs['Localidad'] = locs.LocNombre.str.title()

In [34]:
locs.to_file("chapters/artropofauna/dat/riqueza_localidad.shp")

In [46]:
locs = gpd.read_file("chapters/artropofauna/dat/riqueza_localidad.shp")
fig, ax = plt.subplots(figsize=(32, 16))
locs.plot(
	ax=ax, 
	column="Riqueza", 
	legend=True, 
	cmap='summer', 
	#legend_kwds={'loc': 'upper left'}
)
#plt.show()
plt.savefig("test.png", dpi=300, bbox_inches='tight', format='png')